# Setup and load data

Imports

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import os

Merge csv

In [2]:
DATA_DIR = Path("./dataset_raw")
files = sorted(DATA_DIR.glob("*.csv"))

print("Files found:")
for f in files:
    print(f)


dfs = []

for f in files:
    print(f"Loading {f.name}")
    df = pd.read_csv(
        f,
        sep=",",
        low_memory=False
    )
    dfs.append(df)

dvf = pd.concat(dfs, ignore_index=True)

Files found:
dataset_raw\75_2020.csv
dataset_raw\75_2021.csv
dataset_raw\75_2022.csv
dataset_raw\75_2023.csv
dataset_raw\75_2024.csv
dataset_raw\77_2020.csv
dataset_raw\77_2021.csv
dataset_raw\77_2022.csv
dataset_raw\77_2023.csv
dataset_raw\77_2024.csv
Loading 75_2020.csv
Loading 75_2021.csv
Loading 75_2022.csv
Loading 75_2023.csv
Loading 75_2024.csv
Loading 77_2020.csv
Loading 77_2021.csv
Loading 77_2022.csv
Loading 77_2023.csv
Loading 77_2024.csv


In [3]:
print("Total rows:", len(dvf))
dvf.head()

Total rows: 20000


,date_mutation,nature_mutation,type_local,code_postal,longitude,latitude,valeur_fonciere,surface_reelle_bati,nombre_pieces_principales,surface_terrain,lot1_numero,lot1_surface_carrez,lot2_numero,lot2_surface_carrez,lot3_numero,lot3_surface_carrez,lot4_numero,lot4_surface_carrez,lot5_numero,lot5_surface_carrez
0,2020-02-23,Vente,Maison,75014,2.260271,48.859397,1.632421e+06,137.182275,2,98.191648,NaN,NaN,NaN,NaN,943.0,NaN,NaN,NaN,347.0,NaN
1,2020-09-15,Vente,Maison,7508,2.234218,48.874561,3.315840e+05,35.240284,3,241.989015,NaN,NaN,NaN,NaN,NaN,NaN,118.0,NaN,NaN,NaN
2,2020-01-15,Vente,Maison,7504,2.333487,48.870736,7.951677e+05,68.797445,3,220.936995,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN
3,2020-08-29,Vente,Maison,7505,2.255946,48.823280,1.352644e+06,117.663009,1,111.807118,NaN,NaN,NaN,NaN,687.0,NaN,NaN,NaN,406.0,NaN
4,2020-09-30,Vente,Maison,7502,2.345774,48.820920,1.214444e+06,146.811171,4,250.293210,NaN,NaN,891.0,NaN,NaN,NaN,631.0,NaN,NaN,NaN


In [4]:
print("Columns:")
print(dvf.columns.tolist())

Columns:
['date_mutation', 'nature_mutation', 'type_local', 'code_postal', 'longitude', 'latitude', 'valeur_fonciere', 'surface_reelle_bati', 'nombre_pieces_principales', 'surface_terrain', 'lot1_numero', 'lot1_surface_carrez', 'lot2_numero', 'lot2_surface_carrez', 'lot3_numero', 'lot3_surface_carrez', 'lot4_numero', 'lot4_surface_carrez', 'lot5_numero', 'lot5_surface_carrez']


## 4.1 filtering

In [5]:
# Keep only sales
dvf = dvf[dvf["nature_mutation"] == "Vente"]

# Keep only the property types we want
dvf = dvf[dvf["type_local"].isin(["Maison", "Appartement"])]

# Optional: remove non-sale artifacts like very small price
dvf = dvf[dvf["valeur_fonciere"] > 1000] 

## 4.2

In [6]:
# Rules: houses need surface_terrain, apartments do not
mask_maison = dvf["type_local"] == "Maison"
mask_appart = dvf["type_local"] == "Appartement"

# Drop rows with missing critical columns
dvf = dvf[~((mask_maison) & 
            dvf[["valeur_fonciere","surface_reelle_bati","nombre_pieces_principales","surface_terrain"]].isna().any(axis=1))]
dvf = dvf[~((mask_appart) & 
            dvf[["valeur_fonciere","surface_reelle_bati","nombre_pieces_principales"]].isna().any(axis=1))]

# Remove unrealistic zeros
dvf = dvf[~((mask_maison) & ((dvf["surface_reelle_bati"] <= 10) | 
                              (dvf["surface_terrain"] <= 10) | 
                              (dvf["nombre_pieces_principales"] <= 0)))]
dvf = dvf[~((mask_appart) & ((dvf["surface_reelle_bati"] <= 5) | 
                              (dvf["nombre_pieces_principales"] <= 0)))]

In [7]:
# Compute price per m² for each transaction (ignore NaN values for surface_reelle_bati)
dvf["prix_m2"] = dvf["valeur_fonciere"] / dvf["surface_reelle_bati"]

# Calculate the average price per m² by postal code (code_postal)
dvf["prix_m2_ref"] = dvf.groupby("code_postal")["prix_m2"].transform("mean")

# Inspect the first few rows to verify the new column 'prix_m2_ref'
print(dvf[["code_postal", "valeur_fonciere", "surface_reelle_bati", "prix_m2", "prix_m2_ref"]].head())

   code_postal  valeur_fonciere  surface_reelle_bati       prix_m2  \
0        75014     1.632421e+06           137.182275  11899.646738   
1         7508     3.315840e+05            35.240284   9409.231780   
2         7504     7.951677e+05            68.797445  11558.099058   
3         7505     1.352644e+06           117.663009  11495.911231   
4         7502     1.214444e+06           146.811171   8272.150535   

    prix_m2_ref  
0  10062.666974  
1   9848.317032  
2  10257.726067  
3   9974.401555  
4   9569.916938  


## 4.3

In [8]:
num_cols_maison = ["valeur_fonciere","surface_reelle_bati","surface_terrain","nombre_pieces_principales"]
num_cols_appart = ["valeur_fonciere","surface_reelle_bati","nombre_pieces_principales"]

# Function to trim with enhanced outlier detection (IQR-based)
def trim_outliers(df, mask, cols):
    for col in cols:
        # Quantile-based trimming (2.5% to 97.5%)
        lower_q = df.loc[mask, col].quantile(0.025)
        upper_q = df.loc[mask, col].quantile(0.975)
        
        # IQR-based trimming (more aggressive)
        Q1 = df.loc[mask, col].quantile(0.25)
        Q3 = df.loc[mask, col].quantile(0.75)
        IQR = Q3 - Q1
        lower_iqr = Q1 - 1.5 * IQR
        upper_iqr = Q3 + 1.5 * IQR
        
        # Use the more aggressive bounds
        lower = max(lower_q, lower_iqr)
        upper = min(upper_q, upper_iqr)
        
        initial_count = mask.sum()
        df = df[~(mask & ~df[col].between(lower, upper))]
        final_count = (df["type_local"] == ("Maison" if mask.iloc[0] else "Appartement")).sum()
        removed = initial_count - final_count
        
        print(f"{col} ({'Maison' if mask.iloc[0] else 'Appartement'}): keeping {lower:.0f} – {upper:.0f} (removed {removed} rows)")
    return df

dvf = trim_outliers(dvf, mask_maison, num_cols_maison)
dvf = trim_outliers(dvf, mask_appart, num_cols_appart)

# Additional outlier removal based on price ratios
print("\n=== Additional ratio-based outlier detection ===")
mask_maison = dvf["type_local"] == "Maison"
mask_appart = dvf["type_local"] == "Appartement"

# Remove transactions with abnormal price/m² ratios
price_ratio = dvf["prix_m2"] / dvf["prix_m2_ref"]
price_ratio_lower = price_ratio.quantile(0.05)
price_ratio_upper = price_ratio.quantile(0.95)

initial_len = len(dvf)
dvf = dvf[(price_ratio >= price_ratio_lower) & (price_ratio <= price_ratio_upper)]
print(f"Price ratio filter: removed {initial_len - len(dvf)} rows (keeping ratio {price_ratio_lower:.2f}x to {price_ratio_upper:.2f}x of ref price)")

# Reset masks
mask_maison = dvf["type_local"] == "Maison"
mask_appart = dvf["type_local"] == "Appartement"

valeur_fonciere (Maison): keeping 148228 – 1568360 (removed 883 rows)
surface_reelle_bati (Maison): keeping 36 – 241 (removed 1437 rows)
surface_terrain (Maison): keeping 34 – 4791 (removed 1963 rows)
nombre_pieces_principales (Maison): keeping 1 – 6 (removed 1963 rows)
valeur_fonciere (Appartement): keeping 148868 – 1561932 (removed 608 rows)
surface_reelle_bati (Appartement): keeping 35 – 241 (removed 980 rows)
nombre_pieces_principales (Appartement): keeping 1 – 6 (removed 980 rows)

=== Additional ratio-based outlier detection ===
Price ratio filter: removed 1706 rows (keeping ratio 0.55x to 1.48x of ref price)


C:\Users\Adam\AppData\Local\Temp\ipykernel_18052\3694788672.py:23: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~(mask & ~df[col].between(lower, upper))]
C:\Users\Adam\AppData\Local\Temp\ipykernel_18052\3694788672.py:23: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~(mask & ~df[col].between(lower, upper))]
C:\Users\Adam\AppData\Local\Temp\ipykernel_18052\3694788672.py:23: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~(mask & ~df[col].between(lower, upper))]
C:\Users\Adam\AppData\Local\Temp\ipykernel_18052\3694788672.py:23: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~(mask & ~df[col].between(lower, upper))]
C:\Users\Adam\AppData\Local\Temp\ipykernel_18052\3694788672.py:23: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~(mask & ~df[col].between(lower, upper))]
C:\Users\Adam\AppData\Loc

# 4.4

In [9]:
lot_cols_surface = ["lot1_surface_carrez","lot2_surface_carrez","lot3_surface_carrez","lot4_surface_carrez","lot5_surface_carrez"]
lot_cols_num = ["lot1_numero","lot2_numero","lot3_numero","lot4_numero","lot5_numero"]

dvf.loc[mask_appart, "total_carrez_surface"] = dvf.loc[mask_appart, lot_cols_surface].sum(axis=1, skipna=True)
dvf.loc[mask_appart, "number_of_lots"] = dvf.loc[mask_appart, lot_cols_num].notna().sum(axis=1)

# 4.5 INSEE

In [10]:
# Load the INSEE indices
TMP_DIR = "./tmp_downloads"

insee_path = os.path.join(TMP_DIR, "valeurs_trimestrielles.csv")
insee_df = pd.read_csv(insee_path, sep=";")
insee_df = insee_df.dropna(subset=["Codes"])
insee_df = insee_df.drop("Codes", axis=1)
insee_df = insee_df.rename(columns={"Libellé": "Year-Q", "Indice des prix des logements (neufs et anciens) – Brut – Base 100 en moyenne annuelle 2015": "indice"})
insee_df = insee_df.reset_index(drop=True)
# Inspect the data
insee_df.head()

,Year-Q,indice
0,2020-T1,110.0
1,2020-T2,110.5
2,2020-T3,111.0
3,2020-T4,111.5
4,2021-T1,112.8


In [11]:
# Split 'Year-Q' into 'year' and 'quarter'
insee_df[["year", "quarter"]] = insee_df["Year-Q"].str.split("-T", expand=True)
insee_df["year"] = insee_df["year"].astype(int)
insee_df["quarter"] = insee_df["quarter"].astype(int)

# Convert 'indice' to numeric (float)
insee_df["indice"] = pd.to_numeric(insee_df["indice"], errors="coerce")

# Get the lowest available year from the data (instead of hardcoding 2020)
base_year = insee_df["year"].min()

# Ensure that the base_year exists in the dataset
base_index_row = insee_df[(insee_df["year"] == base_year) & (insee_df["quarter"] == 1)]

# Check if the row for base_year-T1 exists
base_index = base_index_row["indice"].values[0]  # base_index is a float
# Normalize the indices to the base year and quarter (e.g., base_year-T1 = 100)
insee_df["normalized_index"] = insee_df["indice"] / base_index * 100
insee_df["normalized_index"] = insee_df["normalized_index"].round().astype(int)

# Inspect the updated INSEE DataFrame after splitting and normalization
insee_df.head()

,Year-Q,indice,year,quarter,normalized_index
0,2020-T1,110.0,2020,1,100
1,2020-T2,110.5,2020,2,100
2,2020-T3,111.0,2020,3,101
3,2020-T4,111.5,2020,4,101
4,2021-T1,112.8,2021,1,103


In [12]:
# Convert 'date_mutation' to datetime if it's not already
dvf["date_mutation"] = pd.to_datetime(dvf["date_mutation"], errors="coerce")

# Extract 'year' and 'quarter' from 'date_mutation' column in DVF data
dvf["year"] = dvf["date_mutation"].dt.year
dvf["quarter"] = dvf["date_mutation"].dt.quarter

# Merge INSEE data into DVF data based on 'year' and 'quarter'
dvf = dvf.merge(insee_df[["year", "quarter", "normalized_index"]], on=["year", "quarter"], how="left")

# Inspect the merged data
dvf["valeur_fonciere_actualisee"] = np.floor(dvf["valeur_fonciere"] * (dvf["normalized_index"] / 100))
dvf = dvf.drop(columns=["year", "quarter", "normalized_index"])

print(dvf[["valeur_fonciere","valeur_fonciere_actualisee"]].head())
# dvf.to_csv("datasets_prepd/dvf_prepared.csv", index=False)

   valeur_fonciere  valeur_fonciere_actualisee
0     7.951677e+05                    795167.0
1     1.352644e+06                   1366169.0
2     1.214444e+06                   1226588.0
3     8.529767e+05                    861506.0
4     7.649113e+05                    772560.0


# 4.6 M²

In [13]:
# 4.7 Season feature
print("\n=== Creating season feature ===")

# Extract month from date_mutation
dvf["month"] = dvf["date_mutation"].dt.month

# Map months to seasons (Northern hemisphere)
def get_season(month):
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    else:  # 9, 10, 11
        return "autumn"

dvf["season"] = dvf["month"].apply(get_season)
dvf = dvf.drop(columns=["month"])

print(f"Season distribution:")
print(dvf["season"].value_counts())
print(f"\nSeason feature added successfully!")


=== Creating season feature ===
Season distribution:
season
summer    3903
spring    3903
autumn    3858
winter    3687
Name: count, dtype: int64

Season feature added successfully!


# Split datasets

# 4.7 Season Feature

In [14]:
# Add department column from code_postal
dvf["departement"] = dvf["code_postal"].astype(str).str[:2]

# Filter the dataset into Maison and Appartement by department
os.makedirs("datasets_prepd", exist_ok=True)

departments = ["77", "75"]

for dept in departments:
    print(f"\n=== Processing department {dept} ===")
    dvf_dept = dvf[dvf["departement"] == dept]
    
    maison_dept = dvf_dept[dvf_dept["type_local"] == "Maison"]
    appart_dept = dvf_dept[dvf_dept["type_local"] == "Appartement"]
    
    print(f"Dept {dept} - Maisons: {len(maison_dept)}, Appartements: {len(appart_dept)}")
    
    maison_dept.to_csv(f"datasets_prepd/dvf_maison_{dept}.csv", index=False)
    appart_dept.to_csv(f"datasets_prepd/dvf_appart_{dept}.csv", index=False)

print("\nAll datasets saved!")


=== Processing department 77 ===
Dept 77 - Maisons: 4791, Appartements: 3388

=== Processing department 75 ===
Dept 75 - Maisons: 4195, Appartements: 2977

All datasets saved!
